# ProcessGuard: Cardiff runtime malware dataset

Notebook này mô tả dataset, EDA và chạy lại pipeline 4 mô hình trên split `test_set` do Cardiff phát hành. Dataset là **machine-level telemetry của mỗi executable trong sandbox**, không phải đặc trưng theo PID. Paper tham khảo: Rhode, Burnap & Jones (2018), *Early-stage malware prediction using recurrent neural networks*, DOI 10.1016/j.cose.2018.05.010. Nguồn dataset: Cardiff Figshare, DOI 10.17035/d.2018.0050524986.

Tải dữ liệu bằng `python scripts/download_cardiff_dataset.py` trước khi chạy notebook. Không có executable nào được chạy bởi notebook này.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

source = Path('../data/raw/cardiff_data_2.csv')
if not source.exists():
    source = Path('data/raw/cardiff_data_2.csv')
if not source.exists():
    raise FileNotFoundError('Run python scripts/download_cardiff_dataset.py from the project root first')
raw = pd.read_csv(source)
raw.shape, raw.columns.tolist()

In [ ]:
# Basic EDA: class counts and the duration distribution per executable sample.
display(raw.groupby(['sample_id', 'malware']).size().groupby(level='malware').size())
display(raw.groupby('malware')['vector'].agg(['min', 'median', 'max', 'count']))
raw.groupby('malware').size().plot(kind='bar', title='Telemetry rows by label', rot=0)
plt.ylabel('Rows'); plt.show()

In [ ]:
# Apply the project's sample-level cleaning and preserve Cardiff's published split.
from process_guard.cardiff import make_sequences
X_seq, y, is_test, sample_ids, metric_names = make_sequences(source, seconds=5)
print('Sequences:', X_seq.shape, '| train:', int((~is_test).sum()), '| test:', int(is_test.sum()))
print('Labels (0 benign / 1 malicious):', pd.Series(y).value_counts().sort_index().to_dict())
print('Metrics per second:', metric_names)

In [ ]:
# Train Decision Tree, Random Forest, Extra Trees and benign-only Isolation Forest.
from scripts.train_cardiff_models import train
comparison = train(source, seconds=5)
comparison.sort_values('f1', ascending=False)

In [ ]:
# Compare early observation windows and inspect feature importance from the tree models.
early = pd.read_csv('../artifacts/reports/cardiff/cardiff_early_window_metrics.csv') if Path('../artifacts/reports/cardiff/cardiff_early_window_metrics.csv').exists() else pd.read_csv('artifacts/reports/cardiff/cardiff_early_window_metrics.csv')
display(early)
importance_path = Path('../artifacts/reports/cardiff/cardiff_random_forest_feature_importance.csv')
if not importance_path.exists(): importance_path = Path('artifacts/reports/cardiff/cardiff_random_forest_feature_importance.csv')
display(pd.read_csv(importance_path).head(15))

## Diễn giải

Các metric trong notebook chỉ mô tả kết quả trên split và môi trường dataset này. Dataset ghi hoạt động toàn máy khi chạy từng executable trong sandbox Windows 7; không dùng để suy ra một PID cụ thể trong chương trình live. Isolation Forest được fit riêng bằng benign train và có thể có recall thấp. Báo cáo sinh tại `artifacts/reports/cardiff/`; model lưu tại `artifacts/models/cardiff/`.